In [ ]:
%pip install selenium

In [2]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
import time
import re # For regular expressions to clean team names

# English

In [ ]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.firefox.service import Service as FirefoxService # Import FirefoxService
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from webdriver_manager.firefox import GeckoDriverManager # To automatically manage geckodriver

# --- Global lists to store all collected data ---
all_commentary_data_en = []
all_match_stats_data_en = []

# Define the URLs for the seasons you want to scrape for English Premier League
season_urls_en = {
    #"2013-2014": "https://www.flashscore.com/football/england/premier-league-2013-2014/results/",
    #"2014-2015": "https://www.flashscore.com/football/england/premier-league-2014-2015/results/",
    #"2015-2016": "https://www.flashscore.com/football/england/premier-league-2015-2016/results/",
    #"2016-2017": "https://www.flashscore.com/football/england/premier-league-2016-2017/results/",
    #"2017-2018": "https://www.flashscore.com/football/england/premier-league-2017-2018/results/",
    #"2018-2019": "https://www.flashscore.com/football/england/premier-league-2018-2019/results/",
    #"2019-2020": "https://www.flashscore.com/football/england/premier-league-2019-2020/results/",
    #"2020-2021": "https://www.flashscore.com/football/england/premier-league-2020-2021/results/",
    #"2021-2022": "https://www.flashscore.com/football/england/premier-league-2021-2022/results/",
    #"2022-2023": "https://www.flashscore.com/football/england/premier-league-2022-2023/results/",
    #"2023-2024": "https://www.flashscore.com/football/england/premier-league-2023-2024/results/",
    "2024-2025": "https://www.flashscore.com/football/england/premier-league-2024-2025/results/"
}

BATCH_SIZE = 76  # Restart browser after this many matches

# Define the statistics you want to scrape for each half in English.
STATS_TO_SCRAPE_EN = [
    "Shots on target",
    "Shots off target",
    "Blocked Shots",
    "Total shots",
    "Ball Possession",
    "Fouls",
    "Yellow Cards",
    "Offsides",
    "Corner Kicks",
    "Throw-ins",
    "Free Kicks",
    "Goalkeeper Saves"
]

# Function to clean stat names for column headers
def clean_stat_name(stat_name):
    cleaned_name = stat_name.replace(' ', '_')
    cleaned_name = re.sub(r'[^a-zA-Z0-9_]', '', cleaned_name)
    cleaned_name = cleaned_name.replace('Perc', '_Perc') # Keep this if 'Perc' is a common suffix you want to delineate
    cleaned_name = re.sub(r'__+', '_', cleaned_name)
    return cleaned_name.strip('_')

# --- Helper function for scraping half stats ---
# This function will now accept current_match_stats as an argument
# so it can update the dictionary for the current match.
def scrape_half_stats(half_url, half_prefix, stats_list, driver, wait, current_match_stats):
    print(f"  Scraping {half_prefix} stats from: {half_url} (English)")
    driver.get(half_url)

    try:
        # Wait for at least one of the main statistic rows to be present
        # Use a more robust contains for the main row and category
        wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'wcl-row_')]/div[contains(@class, 'wcl-category_')]")))
        time.sleep(1) # Small additional pause for content to settle

        # Find all individual stat category blocks (e.g., one for 'Ball Possession', one for 'Goal Attempts')
        # Use a more robust contains for the main row and category
        stat_category_blocks = driver.find_elements(By.XPATH, "//div[contains(@class, 'wcl-row_')]/div[contains(@class, 'wcl-category_')]")

        if not stat_category_blocks:
            print(f"  Warning: No statistic category blocks found for {half_prefix} at {half_url}.")
            for stat_name in stats_list:
                cleaned_stat_name = clean_stat_name(stat_name)
                current_match_stats[f"{half_prefix}_Home_{cleaned_stat_name}"] = "N/A"
                current_match_stats[f"{half_prefix}_Away_{cleaned_stat_name}"] = "N/A"
            return # Exit if no blocks are found

        # Iterate through each stat name we want to scrape
        for stat_name_to_find in stats_list:
            home_val = "0"
            away_val = "0"
            stat_found_and_processed = False

            # Iterate through the found HTML elements that represent statistic categories
            for stat_block_element in stat_category_blocks:
                try:
                    # Try to find the category name div within the current stat block
                    # Use a more robust contains for the category name
                    category_name_element = stat_block_element.find_element(By.XPATH, ".//div[contains(@class, 'wcl-category_')]")

                    # Check if the text of the category name matches the stat we are looking for
                    if category_name_element.text.strip() == stat_name_to_find:
                        # Found the correct stat row, now extract home and away values
                        try:
                            # Target the <strong> tag directly within the home/away value divs
                            # Use more robust contains for home and away value divs
                            home_value_element = stat_block_element.find_element(By.XPATH, ".//div[contains(@class, 'wcl-homeValue_')]//strong")
                            away_value_element = stat_block_element.find_element(By.XPATH, ".//div[contains(@class, 'wcl-awayValue_')]//strong")

                            home_val = home_value_element.text.strip()
                            away_val = away_value_element.text.strip()
                            stat_found_and_processed = True
                            break # Break out of the inner loop (stat_block_elements) as we found the stat
                        except NoSuchElementException:
                            print(f"    Warning: Value strong tag not found for '{stat_name_to_find}' ({half_prefix}). Setting to ERROR_VAL_MISSING.")
                            home_val = "ERROR_VAL_MISSING"
                            away_val = "ERROR_VAL_MISSING"
                            stat_found_and_processed = True
                            break # Break even if value is missing, as the stat row itself was found
                        except Exception as inner_e:
                            print(f"    Error extracting values for '{stat_name_to_find}' ({half_prefix}): {inner_e}. Setting to ERROR_EXTRACTION.")
                            home_val = "ERROR_EXTRACTION"
                            away_val = "ERROR_EXTRACTION"
                            stat_found_and_processed = True
                            break # Break on other errors too

                except NoSuchElementException:
                    # This specific stat_block_element might not contain a category div,
                    # or it's not the one we're looking for. Continue to the next block.
                    pass
                except Exception as block_e:
                    print(f"    Error processing a stat block for '{stat_name_to_find}' ({half_prefix}): {block_e}")
                    # Don't set to ERROR here, as it might be a transient issue with one block,
                    # and the stat itself might be found in another block.
                    # If it's not found in any block, it will remain N/A.

            # After checking all stat_block_elements for the current stat_name_to_find
            cleaned_stat_name = clean_stat_name(stat_name_to_find)
            current_match_stats[f"{half_prefix}_Home_{cleaned_stat_name}"] = home_val
            current_match_stats[f"{half_prefix}_Away_{cleaned_stat_name}"] = away_val

    except TimeoutException:
        print(f"  Timeout waiting for {half_prefix} stats at {half_url} (English). Populating with N/A.")
        for stat_name in stats_list:
            cleaned_stat_name = clean_stat_name(stat_name)
            current_match_stats[f"{half_prefix}_Home_{cleaned_stat_name}"] = "N/A"
            current_match_stats[f"{half_prefix}_Away_{cleaned_stat_name}"] = "N/A"
    except Exception as e:
        print(f"  An unexpected error occurred during scraping {half_prefix} stats from {half_url}: {e}. Populating with ERROR_OVERALL.")
        for stat_name in stats_list:
            cleaned_stat_name = clean_stat_name(stat_name)
            current_match_stats[f"{half_prefix}_Home_{cleaned_stat_name}"] = "ERROR_OVERALL"
            current_match_stats[f"{half_prefix}_Away_{cleaned_stat_name}"] = "ERROR_OVERALL"

# --- Main Scraping Loop by Season ---
for season, base_url in season_urls_en.items():
    print(f"\n--- Starting to scrape season: {season} (English) ---")

    # Initialize driver for season results page
    service = FirefoxService(GeckoDriverManager().install()) # Use FirefoxService and GeckoDriverManager
    driver = webdriver.Firefox(service=service) # Use webdriver.Firefox
    wait = WebDriverWait(driver, 3) # Short wait for initial page loads

    driver.get(base_url)

    # Step 1: Click "show more" repeatedly to load all matches
    while True:
        try:
            # Look for the specific "show more" button for the season results table
            show_more = wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="live-table"]/div[1]/div/div/a'))
            )
            driver.execute_script("arguments[0].click();", show_more)
            time.sleep(2)  # Wait for content to load after clicking
        except (NoSuchElementException, TimeoutException):
            print(f"No more 'show more' button found or timed out for {season}. All matches loaded.")
            break
        except Exception as e:
            print(f"An unexpected error occurred while clicking 'show more' for {season}: {e}")
            break  # Exit loop if unexpected error

    # Step 2: Extract all match links and details from the season results page
    current_season_match_details = []
    try:
        # Wait for at least one match row to be present
        wait.until(
            EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]"))
        )
        match_elements_rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]")

        for match_row_element in match_elements_rows:
            match_link = ""
            home_team = "N/A"
            away_team = "N/A"
            match_date_time = "N/A"  # This will capture "DD.MM. HH:MM"

            try:
                # The match link is typically found in an 'a' tag with class 'eventRowLink'
                link_element = match_row_element.find_element(By.XPATH, ".//a[@class='eventRowLink']")
                href = link_element.get_attribute("href")
                if href and "/#/match-summary" in href: # English URL fragment
                    match_link = href
            except NoSuchElementException:
                pass  # Link not found for this row, skip

            if not match_link:
                continue  # Skip if no valid match link is found

            try:
                # Home team name - adapt XPath if needed based on the current Flashscore DOM
                home_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__homeParticipant')]")
                home_team = home_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                # Away team name - adapt XPath if needed based on the current Flashscore DOM
                away_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__awayParticipant')]")
                away_team = away_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                # Date and Time - adapt XPath if needed
                date_element = match_row_element.find_element(By.XPATH, ".//div[@class='event__time']")
                match_date_time = date_element.text.strip()
            except NoSuchElementException:
                pass

            current_season_match_details.append({
                "Season": season,
                "Match URL": match_link,
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time  # Use this for consistency
            })
    except TimeoutException:
        print(f"Timeout while waiting for match rows for {season}. No matches found or loaded slowly.")
    except Exception as e:
        print(f"An unexpected error occurred while extracting match links for {season}: {e}")

    print(f"Found {len(current_season_match_details)} match links with details for {season}.")
    driver.quit()  # Close the browser after collecting all match links for the season

    # Step 3: Scrape commentaries and statistics in batches
    for i in range(0, len(current_season_match_details), BATCH_SIZE):
        print(f"\n[INFO] Restarting browser... Batch starting from match {i+1} of {len(current_season_match_details)} (English)")

        # Initialize new driver for batch processing
        service = FirefoxService(GeckoDriverManager().install()) # Use FirefoxService and GeckoDriverManager
        driver = webdriver.Firefox(service=service) # Use webdriver.Firefox
        wait = WebDriverWait(driver, 10) # Increased wait time for individual match pages

        batch = current_season_match_details[i:i + BATCH_SIZE]

        for j, match_info in enumerate(batch):
            total_index = i + j
            # Get the base URL (without hash fragment) to construct stats and commentary URLs
            base_match_url = match_info["Match URL"].split('#')[0]

            home_team = match_info["Home Team"]
            away_team = match_info["Away Team"]
            match_date_time = match_info["Match Date Time"]
            current_season_name = match_info["Season"]

            print(f"Processing match {total_index+1}/{len(current_season_match_details)} ({current_season_name}): {home_team} vs {away_team} ({match_date_time})")

            # --- Commentary Scraping ---
            url_for_commentary = base_match_url + "#/match-summary/live-commentary/0"

            current_match_commentary = {
                "Season": current_season_name,
                "Match URL": match_info["Match URL"],
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time,
                "Commentary": ""
            }

            try:
                driver.get(url_for_commentary)
                # Wait for the main commentary section to load
                wait.until(
                    EC.presence_of_element_located((By.XPATH, "//div[@class='section liveCommentary']"))
                )
                time.sleep(2)  # Give a moment for content to fully render after initial load

                # Find all commentary entries
                commentary_entries = driver.find_elements(By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]")

                if commentary_entries:
                    full_commentary_text = "\n".join([entry.text for entry in commentary_entries if entry.text.strip()])
                    if full_commentary_text.strip():
                        current_match_commentary["Commentary"] = full_commentary_text
                    else:
                        print(f"  Found commentary containers but no text for {url_for_commentary}.")
                        current_match_commentary["Commentary"] = "Found containers but no detailed commentary text."
                else:
                    try:
                        # Check for explicit "No commentary" message
                        no_commentary_message = driver.find_element(By.XPATH, "//div[@class='section liveCommentary']//div[contains(text(), 'No hay comentarios')]")
                        commentary_message = no_commentary_message.text.strip()
                        print(f"  No commentary for {url_for_commentary}. Message: '{commentary_message}'")
                        current_match_commentary["Commentary"] = commentary_message
                    except NoSuchElementException:
                        print(f"  No commentary elements or explicit message found for {url_for_commentary}.")
                        current_match_commentary["Commentary"] = "No commentary available for this match."
            except NoSuchElementException as e:
                print(f"  Error finding main container for commentary at {url_for_commentary}: {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Error: Main container not found ({str(e)})."
            except TimeoutException:
                print(f"  Timeout while loading commentary at {url_for_commentary}. Skipping commentary.")
                current_match_commentary["Commentary"] = "Timeout: Could not load commentary."
            except Exception as e:
                print(f"  Unexpected error at {url_for_commentary}: {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Unexpected Error: {str(e)}"

            all_commentary_data_en.append(current_match_commentary)

            # --- Statistics Scraping ---
            # Initialize current_match_stats for THIS specific match
            current_match_stats = {
                "Season": current_season_name,
                "Match URL": match_info["Match URL"],
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time
            }

            # Construct first half and second half URLs
            first_half_stats_url = base_match_url + "#/match-summary/match-statistics/1" # English URL fragment
            second_half_stats_url = base_match_url + "#/match-summary/match-statistics/2" # English URL fragment
            full_time_stats_url = base_match_url + "#/match-summary/match-statistics/0" # English URL fragment
            # Call the modified scrape_half_stats, passing current_match_stats
            scrape_half_stats(first_half_stats_url, "1H", STATS_TO_SCRAPE_EN, driver, wait, current_match_stats)
            time.sleep(1) # Small pause between half scrapes

            scrape_half_stats(second_half_stats_url, "2H", STATS_TO_SCRAPE_EN, driver, wait, current_match_stats)
            time.sleep(1) # Small pause between half scrapes
            
            scrape_half_stats(full_time_stats_url, "FT", STATS_TO_SCRAPE_EN, driver, wait, current_match_stats)
            all_match_stats_data_en.append(current_match_stats)
            time.sleep(2) # Pause before moving to the next match in the batch

        driver.quit()  # Close the browser after each batch

# Step 4: Combine and save the final data
print("\n--- Combining and saving data (English) ---")

df_commentary_en = pd.DataFrame(all_commentary_data_en)
df_stats_en = pd.DataFrame(all_match_stats_data_en)

# Using a robust merge with validation
try:
    merged_df_en = pd.merge(df_commentary_en, df_stats_en,
                         on=['Match URL', 'Season', 'Home Team', 'Away Team', 'Match Date Time'],
                         how='left',
                         suffixes=('_commentary', '_stats'),
                         validate='one_to_one') # Ensures each match has unique commentary and stats
except pd.errors.MergeError as e:
    print(f"Merge Error: {e}. This might indicate duplicate match URLs or mismatch in join keys.")
    print("Attempting merge without validation, but investigate source data for duplicates if issues arise.")
    merged_df_en = pd.merge(df_commentary_en, df_stats_en,
                         on=['Match URL', 'Season', 'Home Team', 'Away Team', 'Match Date Time'],
                         how='left',
                         suffixes=('_commentary', '_stats'))


output_filename_en = "flashscore_premier_league_combined_stats_commentary_EN.csv"
merged_df_en.to_csv(output_filename_en, index=False, encoding='utf-8')

print(f"\nDone: Combined data (commentary and stats) for all seasons (English) saved to {output_filename_en}")
print(f"\nFirst 5 rows of the combined DataFrame (English):")
# Use print for non-Jupyter environments
# display(merged_df_en.head()) # .to_string() for better console output
print(merged_df_en.head().to_string())
print(f"\nShape of the combined DataFrame (English): {merged_df_en.shape}")

# Step 5: Convert stats to text format and add as new columns, then drop original stat columns
def stats_to_text_split(row): # stats_df was not needed as it's applied row by row
    first_half_lines = []
    second_half_lines = []

    match_info = f"Match: {row['Home Team']} vs {row['Away Team']} ({row['Match Date Time']})"

    # First Half Stats Table
    first_half_lines.append(match_info)
    first_half_lines.append("First Half Stats:")
    first_half_lines.append("{:<25} {:>10} - {:<10}".format("Stat", "Home", "Away"))
    for stat in STATS_TO_SCRAPE_EN:
        home_stat_key = f"1H_Home_{clean_stat_name(stat)}"
        away_stat_key = f"1H_Away_{clean_stat_name(stat)}"
        home_stat = row.get(home_stat_key, "N/A") # Use .get() with default for robustness
        away_stat = row.get(away_stat_key, "N/A") # Use .get() with default for robustness
        first_half_lines.append("{:<25} {:>10} - {:<10}".format(stat, home_stat, away_stat))
    first_half_lines.append("\n") # Add a newline to separate from next match's stats in the combined text

    # Second Half Stats Table
    second_half_lines.append(match_info)
    second_half_lines.append("Second Half Stats:")
    second_half_lines.append("{:<25} {:>10} - {:<10}".format("Stat", "Home", "Away"))
    for stat in STATS_TO_SCRAPE_EN:
        home_stat_key = f"2H_Home_{clean_stat_name(stat)}"
        away_stat_key = f"2H_Away_{clean_stat_name(stat)}"
        home_stat = row.get(home_stat_key, "N/A")
        away_stat = row.get(away_stat_key, "N/A")
        second_half_lines.append("{:<25} {:>10} - {:<10}".format(stat, home_stat, away_stat))
    second_half_lines.append("\n") # Add a newline

    return "\n".join(first_half_lines), "\n".join(second_half_lines)


# Apply function and add two new columns
# Using .apply on a DataFrame row by row: row is a Series
merged_df_en[['1st Half Stats', '2nd Half Stats']] = merged_df_en.apply(
    lambda row: stats_to_text_split(row),
    axis=1,
    result_type='expand' # This makes it return two columns
)

# Drop all original stat columns
stats_columns_to_drop = []
for stat in STATS_TO_SCRAPE_EN:
    cleaned_stat = clean_stat_name(stat)
    stats_columns_to_drop.append(f"1H_Home_{cleaned_stat}")
    stats_columns_to_drop.append(f"1H_Away_{cleaned_stat}")
    stats_columns_to_drop.append(f"2H_Home_{cleaned_stat}")
    stats_columns_to_drop.append(f"2H_Away_{cleaned_stat}")

# Filter out columns that don't exist in the DataFrame before dropping
existing_stats_columns_to_drop = [col for col in stats_columns_to_drop if col in merged_df_en.columns]

if existing_stats_columns_to_drop:
    merged_df_en.drop(columns=existing_stats_columns_to_drop, inplace=True)
else:
    print("No original stat columns found to drop. Check column naming if this is unexpected.")

# Save final DataFrame to CSV
output_filename_text_en = "flashscore_premier_league_combined_stats_commentary_split_text_EN.csv"
merged_df_en.to_csv(output_filename_text_en, index=False, encoding='utf-8')

In [ ]:
import shutil
import os
import pandas as pd
import re

# Function to split commentary into halves
def split_by_half(commentary_text):
    if pd.isna(commentary_text) or not isinstance(commentary_text, str) or commentary_text in {"Found containers but no detailed commentary text.", "No commentary available for this match.", "Timeout: Could not load commentary.", "Error: Main container not found", "Unexpected Error"}:
        return "", ""
    
    lines = commentary_text.strip().split('\n')
    
    timestamp_pattern = re.compile(r"^(\d+\+?\d*)'")
    first_half_blocks = []
    second_half_blocks = []
    
    current_block = []
    current_minute = None
    
    def add_block():
        if current_block and current_minute is not None:
            if current_minute <= 45:
                first_half_blocks.extend(current_block)
            else:
                second_half_blocks.extend(current_block)
    
    for line in lines:
        match = timestamp_pattern.match(line)
        if match:
            # Add previous block to the right half
            add_block()
            
            # Start a new block
            current_block = [line]
            minute_raw = match.group(1)
            parts = minute_raw.split('+')
            base = int(parts[0])
            current_minute = base
        else:
            # Add continuation lines (e.g., description lines)
            if current_block:
                current_block.append(line)
    
    # Add the last block
    add_block()
    
    return '\n'.join(first_half_blocks), '\n'.join(second_half_blocks)

# Load the English commentary data
Season_data = pd.read_csv('flashscore_premier_league_combined_stats_commentary_EN.csv') # Changed filename to EN

# Apply split function row-wise
Season_data[['First Half Commentary', 'Second Half Commentary']] = Season_data['Commentary'].apply(
    lambda x: pd.Series(split_by_half(x))
)
Season_data.drop(columns=['Commentary'], inplace=True)

# Optional: Save to CSV
Season_data.to_csv("flashscore_premier_league_combined_stats_commentary_split_text_EN.csv", index=False) # Changed filename to EN

# Define the source and destination file paths
source_file = 'flashscore_premier_league_combined_stats_commentary_split_text_EN.csv' # Changed filename to EN
destination_file = './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2024-25.csv' # Changed filename to EN and added season

# --- Perform the file copy operation ---
try:
    # Create the Commentary_Files directory if it doesn't exist
    os.makedirs(os.path.dirname(destination_file), exist_ok=True)

    # Use shutil.copy2 to copy the file, preserving metadata
    shutil.copy2(source_file, destination_file)
    print(f"Successfully copied '{source_file}' to '{destination_file}'")

    # Optional: Verify if the destination file exists
    if os.path.exists(destination_file):
        print(f"Verification: '{destination_file}' now exists.")
    else:
        print(f"Verification: '{destination_file}' does not seem to exist after copy.")

except FileNotFoundError:
    print(f"Error: The source file '{source_file}' was not found.")
except PermissionError:
    print(f"Error: Permission denied when trying to copy to '{destination_file}'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Spanish

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# Define the URLs for the seasons you want to scrape
season_urls = {
    #"2013-2014": "https://www.flashscore.es/futbol/inglaterra/premier-league-2013-2014/resultados/",
    #"2014-2015": "https://www.flashscore.es/futbol/inglaterra/premier-league-2014-2015/resultados/",
    #"2015-2016": "https://www.flashscore.es/futbol/inglaterra/premier-league-2015-2016/resultados/",
    #"2016-2017": "https://www.flashscore.es/futbol/inglaterra/premier-league-2016-2017/resultados/",
    #"2017-2018": "https://www.flashscore.es/futbol/inglaterra/premier-league-2017-2018/resultados/",
    #"2018-2019": "https://www.flashscore.es/futbol/inglaterra/premier-league-2018-2019/resultados/",
    #"2019-2020": "https://www.flashscore.es/futbol/inglaterra/premier-league-2019-2020/resultados/",
    #"2020-2021": "https://www.flashscore.es/futbol/inglaterra/premier-league-2020-2021/resultados/",
    #"2021-2022": "https://www.flashscore.es/futbol/inglaterra/premier-league-2021-2022/resultados/",
    #"2022-2023": "https://www.flashscore.es/futbol/inglaterra/premier-league-2022-2023/resultados/",
    #"2023-2024": "https://www.flashscore.es/futbol/inglaterra/premier-league-2023-2024/resultados/",
    "2024-2025": "https://www.flashscore.es/futbol/inglaterra/premier-league-2024-2025/resultados/"
}

BATCH_SIZE = 76  # Restart browser after this many matches

# --- Global lists to store all collected data ---
all_commentary_data = []

# --- Main Scraping Loop by Season ---
for season, base_url in season_urls.items():
    print(f"\n--- Starting to scrape season: {season} ---")
    # Use Firefox WebDriver
    driver = webdriver.Firefox()  # Start a new Firefox browser for each season
    wait = WebDriverWait(driver, 3)  # Initialize WebDriverWait for the new driver

    driver.get(base_url)

    # Step 1: Click "show more" repeatedly to load all matches
    while True:
        try:
            # Look for the specific "show more" button for the season results table
            show_more = wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="live-table"]/div[1]/div/div/a'))
            )
            driver.execute_script("arguments[0].click();", show_more)
            time.sleep(2)  # Wait for content to load after clicking
        except (NoSuchElementException, TimeoutException):
            print(f"No more 'show more' button found or timed out for {season}. All matches loaded.")
            break
        except Exception as e:
            print(f"An unexpected error occurred while clicking 'show more' for {season}: {e}")
            break  # Exit loop if unexpected error

    # Step 2: Extract all match links and details from the season results page
    current_season_match_details = []
    try:
        # Wait for at least one match row to be present
        wait.until(
            EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]"))
        )
        match_elements_rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]")

        for match_row_element in match_elements_rows:
            match_link = ""
            home_team = "N/A"
            away_team = "N/A"
            match_date_time = "N/A"  # This will capture "DD.MM. HH:MM"

            try:
                # The match link is typically found in an 'a' tag with class 'eventRowLink'
                link_element = match_row_element.find_element(By.XPATH, ".//a[@class='eventRowLink']")
                href = link_element.get_attribute("href")
                if href and "/#/resumen-del-partido" in href:
                    match_link = href
            except NoSuchElementException:
                pass  # Link not found for this row, skip

            if not match_link:
                continue  # Skip if no valid match link is found

            try:
                # Home team name - adapt XPath if needed based on the current Flashscore DOM
                home_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__homeParticipant')]")
                home_team = home_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                # Away team name - adapt XPath if needed based on the current Flashscore DOM
                away_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__awayParticipant')]")
                away_team = away_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                # Date and Time - adapt XPath if needed
                date_element = match_row_element.find_element(By.XPATH, ".//div[@class='event__time']")
                match_date_time = date_element.text.strip()
            except NoSuchElementException:
                pass

            current_season_match_details.append({
                "Season": season,
                "Match URL": match_link,
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time  # Use this for consistency
            })
    except TimeoutException:
        print(f"Timeout while waiting for match rows for {season}. No matches found or loaded slowly.")
    except Exception as e:
        print(f"An unexpected error occurred while extracting match links for {season}: {e}")

    print(f"Found {len(current_season_match_details)} match links with details for {season}.")
    driver.quit()  # Close the browser after collecting all match links for the season

    # Step 3: Scrape commentaries and statistics in batches
    for i in range(0, len(current_season_match_details), BATCH_SIZE):
        print(f"\n[INFO] Restarting browser... Batch starting from match {i+1} of {len(current_season_match_details)}")
        driver = webdriver.Firefox()  # Start a new Firefox browser for each batch
        wait = WebDriverWait(driver, 3)  # Initialize WebDriverWait for the new driver instance
        batch = current_season_match_details[i:i + BATCH_SIZE]

        for j, match_info in enumerate(batch):
            total_index = i + j
            # Get the base URL (without hash fragment) to construct stats and commentary URLs
            base_match_url = match_info["Match URL"].split('#')[0]

            home_team = match_info["Home Team"]
            away_team = match_info["Away Team"]
            match_date_time = match_info["Match Date Time"]
            current_season_name = match_info["Season"]

            print(f"Processing match {total_index+1}/{len(current_season_match_details)} ({current_season_name}): {home_team} vs {away_team} ({match_date_time})")

            # --- Commentary Scraping ---
            url_for_commentary = base_match_url + "#/resumen-del-partido/comentarios-en-directo/0"

            current_match_commentary = {
                "Season": current_season_name,
                "Match URL": match_info["Match URL"],
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time,
                "Commentary": ""
            }

            try:
                driver.get(url_for_commentary)
                # Wait for the main commentary section to load
                wait.until(
                    EC.presence_of_element_located((By.XPATH, "//div[@class='section liveCommentary']"))
                )
                # Wait for at least one commentary entry to be present
                wait.until(
                    EC.presence_of_all_elements_located((By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]"))
                )
                time.sleep(2)  # Give a moment for content to fully render after initial load

                # Find all commentary entries
                commentary_entries = driver.find_elements(By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]")

                if commentary_entries:
                    full_commentary_text = "\n".join([entry.text for entry in commentary_entries if entry.text.strip()])
                    if full_commentary_text.strip():
                        current_match_commentary["Commentary"] = full_commentary_text
                    else:
                        print(f"  Found commentary containers but no text for {url_for_commentary}.")
                        current_match_commentary["Commentary"] = "Found containers but no detailed commentary text."
                else:
                    try:
                        # Check for explicit "No commentary" message
                        no_commentary_message = driver.find_element(By.XPATH, "//div[@class='section liveCommentary']//div[contains(text(), 'No hay comentarios')]")
                        commentary_message = no_commentary_message.text.strip()
                        print(f"  No commentary for {url_for_commentary}. Message: '{commentary_message}'")
                        current_match_commentary["Commentary"] = commentary_message
                    except NoSuchElementException:
                        print(f"  No commentary elements or explicit message found for {url_for_commentary}.")
                        current_match_commentary["Commentary"] = "No commentary available for this match."
            except NoSuchElementException as e:
                print(f"  Error finding main container for commentary at {url_for_commentary}: {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Error: Main container not found ({str(e)})."
            except TimeoutException:
                print(f"  Timeout while loading commentary at {url_for_commentary}. Skipping commentary.")
                current_match_commentary["Commentary"] = "Timeout: Could not load commentary."
            except Exception as e:
                print(f"  Unexpected error at {url_for_commentary}: {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Unexpected Error: {str(e)}"

            all_commentary_data.append(current_match_commentary)
            time.sleep(2)  # Delay before moving to the next match

        driver.quit()  # Close the browser after each batch

# Step 4: Filter and save only matches with meaningful Spanish commentary
print("\n--- Saving Spanish commentary only ---")

df_commentary = pd.DataFrame(all_commentary_data)

# Drop rows with missing or trivial commentary
df_commentary = df_commentary[df_commentary['Commentary'].notna()]

# Select only relevant columns
spanish_commentary_df = df_commentary[['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'Commentary']]

# Save to CSV
output_filename = "flashscore_premier_league_commentary_ES.csv"
spanish_commentary_df.to_csv(output_filename, index=False, encoding='utf-8')

print(f"\nDone: Saved Spanish commentary for all matches to {output_filename}")
print(f"\nFirst 5 rows of the Spanish commentary DataFrame:")
# This part requires an environment where display is available (e.g., Jupyter Notebook)
# For a standard Python script, you might just print the head
print(spanish_commentary_df.head().to_string())
print(f"\nShape of the Spanish commentary DataFrame: {spanish_commentary_df.shape}")

In [ ]:
#Copy to season specific CSV
import shutil
import os
import pandas as pd
import re

# Function to split commentary into halves
def split_by_half(commentary_text):
    if pd.isna(commentary_text) or not isinstance(commentary_text, str) or commentary_text in {"Found containers but no detailed commentary text.", "No commentary available for this match.", "No hay comentarios", "Timeout: Could not load commentary.", "Error: Main container not found", "Unexpected Error"}:
        return "", ""
    
    lines = commentary_text.strip().split('\n')
    
    timestamp_pattern = re.compile(r"^(\d+\+?\d*)'")
    first_half_blocks = []
    second_half_blocks = []
    
    current_block = []
    current_minute = None
    
    def add_block():
        if current_block and current_minute is not None:
            if current_minute <= 45:
                first_half_blocks.extend(current_block)
            else:
                second_half_blocks.extend(current_block)
    
    for line in lines:
        match = timestamp_pattern.match(line)
        if match:
            # Add previous block to the right half
            add_block()
            
            # Start a new block
            current_block = [line]
            minute_raw = match.group(1)
            parts = minute_raw.split('+')
            base = int(parts[0])
            current_minute = base
        else:
            # Add continuation lines (e.g., description lines)
            if current_block:
                current_block.append(line)
    
    # Add the last block
    add_block()
    
    return '\n'.join(first_half_blocks), '\n'.join(second_half_blocks)
Season_data = pd.read_csv('flashscore_premier_league_commentary_ES.csv')
# Apply split function row-wise
Season_data[['First Half Commentary', 'Second Half Commentary']] = Season_data['Commentary'].apply(
    lambda x: pd.Series(split_by_half(x))
)
Season_data.drop(columns=['Commentary'], inplace=True)
# Optional: Save to CSV
Season_data.to_csv("flashscore_premier_league_combined_stats_commentary_split_text_ES.csv", index=False)


# Define the source and destination file paths
source_file = 'flashscore_premier_league_combined_stats_commentary_split_text_ES.csv'
destination_file = './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2024-25.csv'

# --- Perform the file copy operation ---
try:
    # Use shutil.copy2 to copy the file, preserving metadata
    shutil.copy2(source_file, destination_file)
    print(f"Successfully copied '{source_file}' to '{destination_file}'")

    # Optional: Verify if the destination file exists
    if os.path.exists(destination_file):
        print(f"Verification: '{destination_file}' now exists.")
    else:
        print(f"Verification: '{destination_file}' does not seem to exist after copy.")

except FileNotFoundError:
    print(f"Error: The source file '{source_file}' was not found.")
except PermissionError:
    print(f"Error: Permission denied when trying to copy to '{destination_file}'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# French

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# Define the URLs for the seasons you want to scrape for French Premier League
season_urls_fr = {
    #"2013-2014": "https://www.flashscore.fr/football/angleterre/premier-league-2013-2014/resultats/",
    #"2014-2015": "https://www.flashscore.fr/football/angleterre/premier-league-2014-2015/resultats/",
    #"2015-2016": "https://www.flashscore.fr/football/angleterre/premier-league-2015-2016/resultats/",
    #"2016-2017": "https://www.flashscore.fr/football/angleterre/premier-league-2016-2017/resultats/",
    #"2017-2018": "https://www.flashscore.fr/football/angleterre/premier-league-2017-2018/resultats/",
    #"2018-2019": "https://www.flashscore.fr/football/angleterre/premier-league-2018-2019/resultats/",
    #"2019-2020": "https://www.flashscore.fr/football/angleterre/premier-league-2019-2020/resultats/",
    #"2020-2021": "https://www.flashscore.fr/football/angleterre/premier-league-2020-2021/resultats/",
    #"2021-2022": "https://www.flashscore.fr/football/angleterre/premier-league-2021-2022/resultats/",
    #"2022-2023": "https://www.flashscore.fr/football/angleterre/premier-league-2022-2023/resultats/",
    #"2023-2024": "https://www.flashscore.fr/football/angleterre/premier-league-2023-2024/resultats/",
    "2024-2025": "https://www.flashscore.fr/football/angleterre/premier-league-2024-2025/resultats/"
}

BATCH_SIZE = 76

# --- Global lists to store all collected data ---
all_commentary_data_fr = []

# --- Main Scraping Loop by Season ---
for season, base_url in season_urls_fr.items():
    print(f"\n--- Starting to scrape season: {season} (French) ---")
    # Change to Firefox WebDriver
    driver = webdriver.Firefox()
    wait = WebDriverWait(driver, 3)

    driver.get(base_url)

    # Step 1: Click "show more" repeatedly to load all matches
    while True:
        try:
            show_more = wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="live-table"]/div[1]/div/div/a'))
            )
            driver.execute_script("arguments[0].click();", show_more)
            time.sleep(2)
        except (NoSuchElementException, TimeoutException):
            print(f"No more 'show more' button found or timed out for {season} (French). All matches loaded.")
            break
        except Exception as e:
            print(f"An unexpected error occurred while clicking 'show more' for {season} (French): {e}")
            break

    # Step 2: Extract all match links and details from the season results page
    current_season_match_details = []
    try:
        wait.until(
            EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]"))
        )
        match_elements_rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]")

        for match_row_element in match_elements_rows:
            match_link = ""
            home_team = "N/A"
            away_team = "N/A"
            match_date_time = "N/A"

            try:
                link_element = match_row_element.find_element(By.XPATH, ".//a[@class='eventRowLink']")
                href = link_element.get_attribute("href")
                # Corrected French URL fragment for match summary
                if href and "/#/resume-du-match" in href:
                    match_link = href
            except NoSuchElementException:
                pass

            if not match_link:
                continue

            try:
                home_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__homeParticipant')]")
                home_team = home_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                away_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__awayParticipant')]")
                away_team = away_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                date_element = match_row_element.find_element(By.XPATH, ".//div[@class='event__time']")
                match_date_time = date_element.text.strip()
            except NoSuchElementException:
                pass

            current_season_match_details.append({
                "Season": season,
                "Match URL": match_link,
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time
            })
    except TimeoutException:
        print(f"Timeout while waiting for match rows for {season} (French). No matches found or loaded slowly.")
    except Exception as e:
        print(f"An unexpected error occurred while extracting match links for {season} (French): {e}")

    print(f"Found {len(current_season_match_details)} match links with details for {season} (French).")
    driver.quit()

    # Step 3: Scrape commentaries and statistics in batches
    for i in range(0, len(current_season_match_details), BATCH_SIZE):
        print(f"\n[INFO] Restarting browser... Batch starting from match {i+1} of {len(current_season_match_details)} (French)")
        # Change to Firefox WebDriver
        driver = webdriver.Firefox()
        wait = WebDriverWait(driver, 3)
        batch = current_season_match_details[i:i + BATCH_SIZE]

        for j, match_info in enumerate(batch):
            total_index = i + j
            base_match_url = match_info["Match URL"].split('#')[0]

            home_team = match_info["Home Team"]
            away_team = match_info["Away Team"]
            match_date_time = match_info["Match Date Time"]
            current_season_name = match_info["Season"]

            print(f"Processing match {total_index+1}/{len(current_season_match_details)} ({current_season_name}, French): {home_team} vs {away_team} ({match_date_time})")

            # --- Commentary Scraping ---
            # Corrected French URL fragment for live commentary
            url_for_commentary = base_match_url + "#/resume-du-match/match-en-direct/0"

            current_match_commentary = {
                "Season": current_season_name,
                "Match URL": match_info["Match URL"],
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time,
                "Commentary": ""
            }

            try:
                driver.get(url_for_commentary)
                wait.until(
                    EC.presence_of_element_located((By.XPATH, "//div[@class='section liveCommentary']"))
                )
                time.sleep(2)

                commentary_entries = driver.find_elements(By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]")

                if commentary_entries:
                    full_commentary_text = "\n".join([entry.text for entry in commentary_entries if entry.text.strip()])
                    if full_commentary_text.strip():
                        current_match_commentary["Commentary"] = full_commentary_text
                    else:
                        print(f"  Found commentary containers but no text for {url_for_commentary} (French).")
                        current_match_commentary["Commentary"] = "Found containers but no detailed commentary text."
                else:
                    try:
                        # Corrected French "No commentary" message
                        no_commentary_message = driver.find_element(By.XPATH, "//div[@class='section liveCommentary']//div[contains(text(), 'Pas de commentaires')]")
                        commentary_message = no_commentary_message.text.strip()
                        print(f"  No commentary for {url_for_commentary} (French). Message: '{commentary_message}'")
                        current_match_commentary["Commentary"] = commentary_message
                    except NoSuchElementException:
                        print(f"  No commentary elements or explicit message found for {url_for_commentary} (French).")
                        current_match_commentary["Commentary"] = "No commentary available for this match."
            except NoSuchElementException as e:
                print(f"  Error finding main container for commentary at {url_for_commentary} (French): {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Error: Main container not found ({str(e)})."
            except TimeoutException:
                print(f"  Timeout while loading commentary at {url_for_commentary} (French). Skipping commentary.")
                current_match_commentary["Commentary"] = "Timeout: Could not load commentary."
            except Exception as e:
                print(f"  Unexpected error at {url_for_commentary} (French): {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Unexpected Error: {str(e)}"

            all_commentary_data_fr.append(current_match_commentary)
            time.sleep(2) # Added a small delay here for robustness between matches

        driver.quit()

# Step 4: Filter and save only matches with meaningful French commentary
print("\n--- Saving French commentary only ---")

df_commentary_fr = pd.DataFrame(all_commentary_data_fr)

# Drop rows with missing or trivial commentary, including the French "No commentary" message
df_commentary_fr = df_commentary_fr[df_commentary_fr['Commentary'].notna()]
# Select only relevant columns
french_commentary_df = df_commentary_fr[['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'Commentary']]

# Save to CSV
output_filename_fr = "flashscore_premier_league_commentary_FR.csv"
french_commentary_df.to_csv(output_filename_fr, index=False, encoding='utf-8')

print(f"\nDone: Saved French commentary for all matches to {output_filename_fr}")
print(f"\nFirst 5 rows of the French commentary DataFrame:")
print(french_commentary_df.head().to_string()) # Changed display() to print() for broader compatibility
print(f"\nShape of the French commentary DataFrame: {french_commentary_df.shape}")

In [ ]:
#Copy to season specific CSV
import shutil
import os
import pandas as pd
import re

# Function to split commentary into halves
def split_by_half(commentary_text):
    if pd.isna(commentary_text) or not isinstance(commentary_text, str) or commentary_text in {"Found containers but no detailed commentary text.", "No commentary available for this match.", "Pas de commentaires", "Timeout: Could not load commentary.", "Error: Main container not found", "Unexpected Error"}:
        return "", ""
    
    lines = commentary_text.strip().split('\n')
    
    timestamp_pattern = re.compile(r"^(\d+\+?\d*)'")
    first_half_blocks = []
    second_half_blocks = []
    
    current_block = []
    current_minute = None
    
    def add_block():
        if current_block and current_minute is not None:
            if current_minute <= 45:
                first_half_blocks.extend(current_block)
            else:
                second_half_blocks.extend(current_block)
    
    for line in lines:
        match = timestamp_pattern.match(line)
        if match:
            # Add previous block to the right half
            add_block()
            
            # Start a new block
            current_block = [line]
            minute_raw = match.group(1)
            parts = minute_raw.split('+')
            base = int(parts[0])
            current_minute = base
        else:
            # Add continuation lines (e.g., description lines)
            if current_block:
                current_block.append(line)
    
    # Add the last block
    add_block()
    
    return '\n'.join(first_half_blocks), '\n'.join(second_half_blocks)
Season_data = pd.read_csv('flashscore_premier_league_commentary_FR.csv')
# Apply split function row-wise
Season_data[['First Half Commentary', 'Second Half Commentary']] = Season_data['Commentary'].apply(
    lambda x: pd.Series(split_by_half(x))
)
Season_data.drop(columns=['Commentary'], inplace=True)
# Optional: Save to CSV
Season_data.to_csv("flashscore_premier_league_combined_stats_commentary_split_text_FR.csv", index=False)


# Define the source and destination file paths
source_file = 'flashscore_premier_league_combined_stats_commentary_split_text_FR.csv'
destination_file = './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2024-25.csv'

# --- Perform the file copy operation ---
try:
    # Use shutil.copy2 to copy the file, preserving metadata
    shutil.copy2(source_file, destination_file)
    print(f"Successfully copied '{source_file}' to '{destination_file}'")

    # Optional: Verify if the destination file exists
    if os.path.exists(destination_file):
        print(f"Verification: '{destination_file}' now exists.")
    else:
        print(f"Verification: '{destination_file}' does not seem to exist after copy.")

except FileNotFoundError:
    print(f"Error: The source file '{source_file}' was not found.")
except PermissionError:
    print(f"Error: Permission denied when trying to copy to '{destination_file}'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# German

In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException

# Define the URLs for the seasons you want to scrape for German Premier League
season_urls_de = {
    #"2013-2014": "https://www.flashscore.de/fussball/england/premier-league-2013-2014/ergebnisse/",
    #"2014-2015": "https://www.flashscore.de/fussball/england/premier-league-2014-2015/ergebnisse/",
    #"2015-2016": "https://www.flashscore.de/fussball/england/premier-league-2015-2016/ergebnisse/",
    #"2016-2017": "https://www.flashscore.de/fussball/england/premier-league-2016-2017/ergebnisse/",
    #"2017-2018": "https://www.flashscore.de/fussball/england/premier-league-2017-2018/ergebnisse/",
    #"2018-2019": "https://www.flashscore.de/fussball/england/premier-league-2018-2019/ergebnisse/",
    #"2019-2020": "https://www.flashscore.de/fussball/england/premier-league-2019-2020/ergebnisse/",
    #"2020-2021": "https://www.flashscore.de/fussball/england/premier-league-2020-2021/ergebnisse/",
    #"2021-2022": "https://www.flashscore.de/fussball/england/premier-league-2021-2022/ergebnisse/",
    #"2022-2023": "https://www.flashscore.de/fussball/england/premier-league-2022-2023/ergebnisse/",
    #"2023-2024": "https://www.flashscore.de/fussball/england/premier-league-2023-2024/ergebnisse/",
    "2024-2025": "https://www.flashscore.de/fussball/england/premier-league-2024-2025/ergebnisse/"
}

BATCH_SIZE = 76

# --- Global lists to store all collected data ---
all_commentary_data_de = []

# --- Main Scraping Loop by Season ---
for season, base_url in season_urls_de.items():
    print(f"\n--- Starting to scrape season: {season} (German) ---")
    # Change to Firefox WebDriver
    driver = webdriver.Firefox()
    wait = WebDriverWait(driver, 3)

    driver.get(base_url)

    # Step 1: Click "show more" repeatedly to load all matches
    while True:
        try:
            show_more = wait.until(
                EC.element_to_be_clickable((By.XPATH, '//*[@id="live-table"]/div[1]/div/div/a'))
            )
            driver.execute_script("arguments[0].click();", show_more)
            time.sleep(2)
        except (NoSuchElementException, TimeoutException):
            print(f"No more 'show more' button found or timed out for {season} (German). All matches loaded.")
            break
        except Exception as e:
            print(f"An unexpected error occurred while clicking 'show more' for {season} (German): {e}")
            break

    # Step 2: Extract all match links and details from the season results page
    current_season_match_details = []
    try:
        wait.until(
            EC.presence_of_all_elements_located((By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]"))
        )
        match_elements_rows = driver.find_elements(By.XPATH, "//div[contains(@class, 'event__match--withRowLink')]")

        for match_row_element in match_elements_rows:
            match_link = ""
            home_team = "N/A"
            away_team = "N/A"
            match_date_time = "N/A"

            try:
                link_element = match_row_element.find_element(By.XPATH, ".//a[@class='eventRowLink']")
                href = link_element.get_attribute("href")
                # Corrected German URL fragment for match summary
                if href and "/#/spiel-zusammenfassung" in href:
                    match_link = href
            except NoSuchElementException:
                pass

            if not match_link:
                continue

            try:
                home_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__homeParticipant')]")
                home_team = home_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                away_team_element = match_row_element.find_element(By.XPATH, ".//div[contains(@class, 'event__awayParticipant')]")
                away_team = away_team_element.text.strip()
            except NoSuchElementException:
                pass

            try:
                date_element = match_row_element.find_element(By.XPATH, ".//div[@class='event__time']")
                match_date_time = date_element.text.strip()
            except NoSuchElementException:
                pass

            current_season_match_details.append({
                "Season": season,
                "Match URL": match_link,
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time
            })
    except TimeoutException:
        print(f"Timeout while waiting for match rows for {season} (German). No matches found or loaded slowly.")
    except Exception as e:
        print(f"An unexpected error occurred while extracting match links for {season} (German): {e}")

    print(f"Found {len(current_season_match_details)} match links with details for {season} (German).")
    driver.quit()

    # Step 3: Scrape commentaries and statistics in batches
    for i in range(0, len(current_season_match_details), BATCH_SIZE):
        print(f"\n[INFO] Restarting browser... Batch starting from match {i+1} of {len(current_season_match_details)} (German)")
        # Change to Firefox WebDriver
        driver = webdriver.Firefox()
        wait = WebDriverWait(driver, 3)
        batch = current_season_match_details[i:i + BATCH_SIZE]

        for j, match_info in enumerate(batch):
            total_index = i + j
            base_match_url = match_info["Match URL"].split('#')[0]

            home_team = match_info["Home Team"]
            away_team = match_info["Away Team"]
            match_date_time = match_info["Match Date Time"]
            current_season_name = match_info["Season"]

            print(f"Processing match {total_index+1}/{len(current_season_match_details)} ({current_season_name}, German): {home_team} vs {away_team} ({match_date_time})")

            # --- Commentary Scraping ---
            # Corrected German URL fragment for live commentary
            url_for_commentary = base_match_url + "#/spiel-zusammenfassung/live-kommentar/0"

            current_match_commentary = {
                "Season": current_season_name,
                "Match URL": match_info["Match URL"],
                "Home Team": home_team,
                "Away Team": away_team,
                "Match Date Time": match_date_time,
                "Commentary": ""
            }

            try:
                driver.get(url_for_commentary)
                wait.until(
                    EC.presence_of_element_located((By.XPATH, "//div[@class='section liveCommentary']"))
                )
                time.sleep(2)

                commentary_entries = driver.find_elements(By.XPATH, "//div[@class='section liveCommentary']//div[contains(@class, 'wcl-commentary_PDHM0')]")

                if commentary_entries:
                    full_commentary_text = "\n".join([entry.text for entry in commentary_entries if entry.text.strip()])
                    if full_commentary_text.strip():
                        current_match_commentary["Commentary"] = full_commentary_text
                    else:
                        print(f"  Found commentary containers but no text for {url_for_commentary} (German).")
                        current_match_commentary["Commentary"] = "Found containers but no detailed commentary text."
                else:
                    try:
                        # Corrected German "No commentary" message
                        no_commentary_message = driver.find_element(By.XPATH, "//div[@class='section liveCommentary']//div[contains(text(), 'Kein Live-Kommentar verfügbar')]")
                        commentary_message = no_commentary_message.text.strip()
                        print(f"  No commentary for {url_for_commentary} (German). Message: '{commentary_message}'")
                        current_match_commentary["Commentary"] = commentary_message
                    except NoSuchElementException:
                        print(f"  No commentary elements or explicit message found for {url_for_commentary} (German).")
                        current_match_commentary["Commentary"] = "No commentary available for this match."
            except NoSuchElementException as e:
                print(f"  Error finding main container for commentary at {url_for_commentary} (German): {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Error: Main container not found ({str(e)})."
            except TimeoutException:
                print(f"  Timeout while loading commentary at {url_for_commentary} (German). Skipping commentary.")
                current_match_commentary["Commentary"] = "Timeout: Could not load commentary."
            except Exception as e:
                print(f"  Unexpected error at {url_for_commentary} (German): {str(e)}. Skipping commentary.")
                current_match_commentary["Commentary"] = f"Unexpected Error: {str(e)}"

            all_commentary_data_de.append(current_match_commentary)
            time.sleep(2) # Added a small delay here for robustness between matches
        driver.quit()

# Step 4: Filter and save only matches with meaningful German commentary
print("\n--- Saving German commentary only ---")

df_commentary_de = pd.DataFrame(all_commentary_data_de)

# Drop rows with missing or trivial commentary, including the German "No commentary" message
df_commentary_de = df_commentary_de[df_commentary_de['Commentary'].notna()]
# Select only relevant columns
german_commentary_df = df_commentary_de[['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'Commentary']]

# Save to CSV
output_filename_de = "flashscore_premier_league_commentary_DE.csv"
german_commentary_df.to_csv(output_filename_de, index=False, encoding='utf-8')

print(f"\nDone: Saved German commentary for all matches to {output_filename_de}")
print(f"\nFirst 5 rows of the German commentary DataFrame:")
print(german_commentary_df.head().to_string()) # Changed display() to print() for broader compatibility
print(f"\nShape of the German commentary DataFrame: {german_commentary_df.shape}")

In [ ]:
#Copy to season specific CSV
import shutil
import os
import pandas as pd
import re

# Function to split commentary into halves
def split_by_half(commentary_text):
    if pd.isna(commentary_text) or not isinstance(commentary_text, str) or commentary_text in {"Found containers but no detailed commentary text.", "No commentary available for this match.", "Kein Live-Kommentar verfügbar", "Timeout: Could not load commentary.", "Error: Main container not found", "Unexpected Error"}:
        return "", ""
    lines = commentary_text.strip().split('\n')
    
    timestamp_pattern = re.compile(r"^(\d+\+?\d*)'")
    first_half_blocks = []
    second_half_blocks = []
    
    current_block = []
    current_minute = None
    
    def add_block():
        if current_block and current_minute is not None:
            if current_minute <= 45:
                first_half_blocks.extend(current_block)
            else:
                second_half_blocks.extend(current_block)
    
    for line in lines:
        match = timestamp_pattern.match(line)
        if match:
            # Add previous block to the right half
            add_block()
            
            # Start a new block
            current_block = [line]
            minute_raw = match.group(1)
            parts = minute_raw.split('+')
            base = int(parts[0])
            current_minute = base
        else:
            # Add continuation lines (e.g., description lines)
            if current_block:
                current_block.append(line)
    
    # Add the last block
    add_block()
    
    return '\n'.join(first_half_blocks), '\n'.join(second_half_blocks)
Season_data = pd.read_csv('flashscore_premier_league_commentary_DE.csv')
# Apply split function row-wise
Season_data[['First Half Commentary', 'Second Half Commentary']] = Season_data['Commentary'].apply(
    lambda x: pd.Series(split_by_half(x))
)
Season_data.drop(columns=['Commentary'], inplace=True)
# Optional: Save to CSV
Season_data.to_csv("flashscore_premier_league_combined_stats_commentary_split_text_DE.csv", index=False)


# Define the source and destination file paths
source_file = 'flashscore_premier_league_combined_stats_commentary_split_text_DE.csv'
destination_file = './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2024-25.csv'

# --- Perform the file copy operation ---
try:
    # Use shutil.copy2 to copy the file, preserving metadata
    shutil.copy2(source_file, destination_file)
    print(f"Successfully copied '{source_file}' to '{destination_file}'")

    # Optional: Verify if the destination file exists
    if os.path.exists(destination_file):
        print(f"Verification: '{destination_file}' now exists.")
    else:
        print(f"Verification: '{destination_file}' does not seem to exist after copy.")

except FileNotFoundError:
    print(f"Error: The source file '{source_file}' was not found.")
except PermissionError:
    print(f"Error: Permission denied when trying to copy to '{destination_file}'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# Combining all the files to make ONE cohesive Dataset

## Step 1. Combine all the English Commentary Files into one File

In [1]:
import pandas as pd
import re
#Part 1: Store each of the 11 files as a separate DataFrame
# List of file names to read
file_names = [
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2013-14.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2014-15.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2015-16.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2016-17.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2017-18.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2018-19.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2019-20.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2020-21.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2021-22.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2022-23.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2023-24.csv']
# Dictionary to hold DataFrames
dataframes = {}
# Read each file into a DataFrame and store it in the dictionary
for file_name in file_names:
    try:
        df = pd.read_csv(file_name, encoding='utf-8')
        season_key = file_name.split('_')[-1].replace('.csv', '')  # Extract season from file name
        dataframes[season_key] = df
        print(f"Successfully loaded {file_name} with shape {df.shape}")
    except FileNotFoundError:
        print(f"File {file_name} not found. Skipping this file.")
    except pd.errors.EmptyDataError:
        print(f"File {file_name} is empty. Skipping this file.")
    except Exception as e:
        print(f"An error occurred while reading {file_name}: {e}")
# Part 2: Make sure there are only Letters in the team names rather than numbers
def clean_team_name(team_name):
    # Remove any non-letter characters (including numbers and special characters)
    cleaned_name = re.sub(r'[^a-zA-Z\s]', '', team_name)
    return cleaned_name.strip()
# Clean team names in each DataFrame
for season, df in dataframes.items():
    if 'Home Team' in df.columns:
        df['Home Team'] = df['Home Team'].apply(clean_team_name)
    if 'Away Team' in df.columns:
        df['Away Team'] = df['Away Team'].apply(clean_team_name)
    print(f"Cleaned team names for season {season}.")
# Part 3: Merge all DataFrames into one
merged_df_en = pd.concat(dataframes.values(), ignore_index=True)
# Ensure the merged DataFrame has the expected columns
expected_columns = ['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'First Half Commentary', 'Second Half Commentary']
for col in expected_columns:
    if col not in merged_df_en.columns:
        merged_df_en[col] = None  # Add missing columns with None values
# Display the first few rows of the merged DataFrame
print("\nMerged DataFrame (English):")
display(merged_df_en.tail())
# Save the merged DataFrame to a CSV file
merged_df_en.to_csv('flashscore_premier_league_combined_stats_commentary_split_text_EN_full_dataset.csv', index=False, encoding='utf-8')

Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2013-14.csv with shape (380, 79)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2014-15.csv with shape (380, 79)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2015-16.csv with shape (380, 79)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2016-17.csv with shape (380, 79)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2017-18.csv with shape (380, 79)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2018-19.csv with shape (380, 79)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_EN_season_2019-20.csv with s

,Season,Match URL,Home Team,Away Team,Match Date Time,1H_Home_Shots_on_target,1H_Away_Shots_on_target,1H_Home_Shots_off_target,1H_Away_Shots_off_target,1H_Home_Blocked_Shots,...,FT_Home_Corner_Kicks,FT_Away_Corner_Kicks,FT_Home_Throwins,FT_Away_Throwins,FT_Home_Free_Kicks,FT_Away_Free_Kicks,FT_Home_Goalkeeper_Saves,FT_Away_Goalkeeper_Saves,First Half Commentary,Second Half Commentary
4175,2023-2024,https://www.flashscore.com/match/football/Sd9u...,Brighton,Luton,12.08. 07:00,3.0,1.0,12.0,5.0,0.0,...,6.0,7.0,9.0,10.0,10.0,6.0,2.0,8.0,45+2'\nDavid Coote blows his whistle to signal...,90+8'\nThat's all for today as the game is ove...
4176,2023-2024,https://www.flashscore.com/match/football/6m8q...,Everton,Fulham,12.08. 07:00,4.0,0.0,5.0,1.0,0.0,...,10.0,4.0,27.0,22.0,9.0,18.0,1.0,9.0,45+2'\nThe end of the first half.\n45+1'\nTher...,90+7'\nStuart Attwell is looking at his watch ...
4177,2023-2024,https://www.flashscore.com/match/football/hjTJ...,Sheffield Utd,Crystal Palace,12.08. 07:00,1.0,3.0,3.0,11.0,0.0,...,5.0,5.0,16.0,33.0,13.0,20.0,7.0,1.0,45+2'\nJohn Brooks has ended the first half by...,90+6'\nThe referee blows for the end of today'...
4178,2023-2024,https://www.flashscore.com/match/football/KW5X...,Arsenal,Nottingham,12.08. 04:30,3.0,0.0,4.0,1.0,0.0,...,8.0,3.0,18.0,9.0,13.0,14.0,1.0,5.0,45+6'\nThe match has reached half-time.\n45+5'...,90+8'\nThat's the end of the match.\n90+6'\nBu...
4179,2023-2024,https://www.flashscore.com/match/football/EkT4...,Burnley,Manchester City,11.08. 12:00,1.0,3.0,3.0,2.0,0.0,...,6.0,5.0,15.0,13.0,9.0,11.0,5.0,1.0,45+6'\nThe match has reached half-time.\n45+5'...,90+9'\nThe referee checks his watch and blows ...


## Step 2. Do the same process for the Spanish, French, and German files

In [2]:
#Spanish
import pandas as pd
import re
#Part 1: Store each of the 11 files as a separate DataFrame
# List of file names to read
file_names = [
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2013-14.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2014-15.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2015-16.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2016-17.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2017-18.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2018-19.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2019-20.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2020-21.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2021-22.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2022-23.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2023-24.csv'
]
# Dictionary to hold DataFrames
dataframes = {}
# Read each file into a DataFrame and store it in the dictionary
for file_name in file_names:
    try:
        df = pd.read_csv(file_name, encoding='utf-8')
        season_key = file_name.split('_')[-1].replace('.csv', '')  # Extract season from file name
        dataframes[season_key] = df
        print(f"Successfully loaded {file_name} with shape {df.shape}")
    except FileNotFoundError:
        print(f"File {file_name} not found. Skipping this file.")
    except pd.errors.EmptyDataError:
        print(f"File {file_name} is empty. Skipping this file.")
    except Exception as e:
        print(f"An error occurred while reading {file_name}: {e}")
# Part 2: Make sure there are only Letters in the team names rather than numbers
def clean_team_name(team_name):
    # Remove any non-letter characters (including numbers and special characters)
    cleaned_name = re.sub(r'[^a-zA-Z\s]', '', team_name)
    return cleaned_name.strip()
# Clean team names in each DataFrame
for season, df in dataframes.items():
    if 'Home Team' in df.columns:
        df['Home Team'] = df['Home Team'].apply(clean_team_name)
    if 'Away Team' in df.columns:
        df['Away Team'] = df['Away Team'].apply(clean_team_name)
    print(f"Cleaned team names for season {season}.")
# Part 3: Merge all DataFrames into one
merged_df_en = pd.concat(dataframes.values(), ignore_index=True)
# Ensure the merged DataFrame has the expected columns
expected_columns = ['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'First Half Commentary', 'Second Half Commentary']
for col in expected_columns:
    if col not in merged_df_en.columns:
        merged_df_en[col] = None  # Add missing columns with None values
# Display the first few rows of the merged DataFrame
print("\nMerged DataFrame (Spanish):")
print(merged_df_en.head().to_string())
# Save the merged DataFrame to a CSV file
merged_df_en.to_csv('flashscore_premier_league_combined_stats_commentary_split_text_ES_full_dataset.csv', index=False, encoding='utf-8')

Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2013-14.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2014-15.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2015-16.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2016-17.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2017-18.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2018-19.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_ES_2019-20.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/fl

In [3]:
#French
import pandas as pd
import re
#Part 1: Store each of the 11 files as a separate DataFrame
# List of file names to read
file_names = [
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2013-14.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2014-15.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2015-16.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2016-17.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2017-18.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2018-19.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2019-20.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2020-21.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2021-22.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2022-23.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2023-24.csv'
]
# Dictionary to hold DataFrames
dataframes = {}
# Read each file into a DataFrame and store it in the dictionary
for file_name in file_names:
    try:
        df = pd.read_csv(file_name, encoding='utf-8')
        season_key = file_name.split('_')[-1].replace('.csv', '')  # Extract season from file name
        dataframes[season_key] = df
        print(f"Successfully loaded {file_name} with shape {df.shape}")
    except FileNotFoundError:
        print(f"File {file_name} not found. Skipping this file.")
    except pd.errors.EmptyDataError:
        print(f"File {file_name} is empty. Skipping this file.")
    except Exception as e:
        print(f"An error occurred while reading {file_name}: {e}")
# Part 2: Make sure there are only Letters in the team names rather than numbers
def clean_team_name(team_name):
    # Remove any non-letter characters (including numbers and special characters)
    cleaned_name = re.sub(r'[^a-zA-Z\s]', '', team_name)
    return cleaned_name.strip()
# Clean team names in each DataFrame
for season, df in dataframes.items():
    if 'Home Team' in df.columns:
        df['Home Team'] = df['Home Team'].apply(clean_team_name)
    if 'Away Team' in df.columns:
        df['Away Team'] = df['Away Team'].apply(clean_team_name)
    print(f"Cleaned team names for season {season}.")
# Part 3: Merge all DataFrames into one
merged_df_en = pd.concat(dataframes.values(), ignore_index=True)
# Ensure the merged DataFrame has the expected columns
expected_columns = ['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'First Half Commentary', 'Second Half Commentary']
for col in expected_columns:
    if col not in merged_df_en.columns:
        merged_df_en[col] = None  # Add missing columns with None values
# Display the first few rows of the merged DataFrame
print("\nMerged DataFrame (French):")
print(merged_df_en.head().to_string())
# Save the merged DataFrame to a CSV file
merged_df_en.to_csv('flashscore_premier_league_combined_stats_commentary_split_text_FR_full_dataset.csv', index=False, encoding='utf-8')

Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2013-14.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2014-15.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2015-16.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2016-17.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2017-18.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2018-19.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_FR_2019-20.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/fl

In [4]:
#German
import pandas as pd
import re
#Part 1: Store each of the 11 files as a separate DataFrame
# List of file names to read
file_names = [
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2013-14.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2014-15.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2015-16.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2016-17.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2017-18.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2018-19.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2019-20.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2020-21.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2021-22.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2022-23.csv',
    './Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2023-24.csv'
]
# Dictionary to hold DataFrames
dataframes = {}
# Read each file into a DataFrame and store it in the dictionary
for file_name in file_names:
    try:
        df = pd.read_csv(file_name, encoding='utf-8')
        season_key = file_name.split('_')[-1].replace('.csv', '')  # Extract season from file name
        dataframes[season_key] = df
        print(f"Successfully loaded {file_name} with shape {df.shape}")
    except FileNotFoundError:
        print(f"File {file_name} not found. Skipping this file.")
    except pd.errors.EmptyDataError:
        print(f"File {file_name} is empty. Skipping this file.")
    except Exception as e:
        print(f"An error occurred while reading {file_name}: {e}")
# Part 2: Make sure there are only Letters in the team names rather than numbers
def clean_team_name(team_name):
    # Remove any non-letter characters (including numbers and special characters)
    cleaned_name = re.sub(r'[^a-zA-Z\s]', '', team_name)
    return cleaned_name.strip()
# Clean team names in each DataFrame
for season, df in dataframes.items():
    if 'Home Team' in df.columns:
        df['Home Team'] = df['Home Team'].apply(clean_team_name)
    if 'Away Team' in df.columns:
        df['Away Team'] = df['Away Team'].apply(clean_team_name)
    print(f"Cleaned team names for season {season}.")
# Part 3: Merge all DataFrames into one
merged_df_en = pd.concat(dataframes.values(), ignore_index=True)
# Ensure the merged DataFrame has the expected columns
expected_columns = ['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time', 'First Half Commentary', 'Second Half Commentary']
for col in expected_columns:
    if col not in merged_df_en.columns:
        merged_df_en[col] = None  # Add missing columns with None values
# Display the first few rows of the merged DataFrame
print("\nMerged DataFrame (Spanish):")
print(merged_df_en.head().to_string())
# Save the merged DataFrame to a CSV file
merged_df_en.to_csv('flashscore_premier_league_combined_stats_commentary_split_text_DE_full_dataset.csv', index=False, encoding='utf-8')

Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2013-14.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2014-15.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2015-16.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2016-17.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2017-18.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2018-19.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/flashscore_premier_league_combined_stats_commentary_split_text_DE_2019-20.csv with shape (380, 7)
Successfully loaded ./Commentary_Files/fl

## Step 3. Combine the 4 Files to make the cohesive dataset

In [5]:
import pandas as pd

# List of file names to combine
file_names = [
    'flashscore_premier_league_combined_stats_commentary_split_text_EN_full_dataset.csv',
    'flashscore_premier_league_combined_stats_commentary_split_text_ES_full_dataset.csv',
    'flashscore_premier_league_combined_stats_commentary_split_text_FR_full_dataset.csv',
    'flashscore_premier_league_combined_stats_commentary_split_text_DE_full_dataset.csv'
]

# Load each file into a DataFrame and store them in a list
dataframes = []
for file_name in file_names:
    try:
        df = pd.read_csv(file_name, encoding='utf-8')
        dataframes.append(df)
        print(f"Successfully loaded {file_name} with shape {df.shape}")
    except FileNotFoundError:
        print(f"File {file_name} not found. Skipping this file.")
    except pd.errors.EmptyDataError:
        print(f"File {file_name} is empty. Skipping this file.")
    except Exception as e:
        print(f"An error occurred while reading {file_name}: {e}")

# Combine all DataFrames into one
combined_df = pd.concat(dataframes, ignore_index=True)

# Save the combined DataFrame to a new CSV file
output_filename = 'flashscore_premier_league_combined_stats_commentary_all_languages_full_dataset.csv'
combined_df.to_csv(output_filename, index=False, encoding='utf-8')
print("\nCombined DataFrame:")
display(combined_df.head().to_string())
print(f"\nShape of the combined DataFrame: {combined_df.shape}") #380*11*4 = 16,760 rows
print(f"Combined dataset saved to {output_filename}")

Successfully loaded flashscore_premier_league_combined_stats_commentary_split_text_EN_full_dataset.csv with shape (4180, 79)
Successfully loaded flashscore_premier_league_combined_stats_commentary_split_text_ES_full_dataset.csv with shape (4180, 7)
Successfully loaded flashscore_premier_league_combined_stats_commentary_split_text_FR_full_dataset.csv with shape (4180, 7)
Successfully loaded flashscore_premier_league_combined_stats_commentary_split_text_DE_full_dataset.csv with shape (4180, 7)

Combined DataFrame:


"      Season                                                           Match URL        Home Team       Away Team Match Date Time  1H_Home_Shots_on_target  1H_Away_Shots_on_target  1H_Home_Shots_off_target  1H_Away_Shots_off_target  1H_Home_Blocked_Shots  1H_Away_Blocked_Shots  1H_Home_Total_shots  1H_Away_Total_shots 1H_Home_Ball_Possession 1H_Away_Ball_Possession  1H_Home_Fouls  1H_Away_Fouls  1H_Home_Yellow_Cards  1H_Away_Yellow_Cards  1H_Home_Offsides  1H_Away_Offsides  1H_Home_Corner_Kicks  1H_Away_Corner_Kicks  1H_Home_Throwins  1H_Away_Throwins  1H_Home_Free_Kicks  1H_Away_Free_Kicks  1H_Home_Goalkeeper_Saves  1H_Away_Goalkeeper_Saves  2H_Home_Shots_on_target  2H_Away_Shots_on_target  2H_Home_Shots_off_target  2H_Away_Shots_off_target  2H_Home_Blocked_Shots  2H_Away_Blocked_Shots  2H_Home_Total_shots  2H_Away_Total_shots 2H_Home_Ball_Possession 2H_Away_Ball_Possession  2H_Home_Fouls  2H_Away_Fouls  2H_Home_Yellow_Cards  2H_Away_Yellow_Cards  2H_Home_Offsides  2H_Away_Offsides  


Shape of the combined DataFrame: (16720, 79)
Combined dataset saved to flashscore_premier_league_combined_stats_commentary_all_languages_full_dataset.csv


In [6]:
import pandas as pd

# File paths
file1 = 'flashscore_premier_league_combined_stats_commentary_split_text_EN_full_dataset.csv'
file2 = 'flashscore_premier_league_combined_stats_commentary_split_text_ES_full_dataset.csv'
file3 = 'flashscore_premier_league_combined_stats_commentary_split_text_FR_full_dataset.csv'
file4 = 'flashscore_premier_league_combined_stats_commentary_split_text_DE_full_dataset.csv'

# 1. Keep all columns from first
df1 = pd.read_csv(file1)

# 2. From subsequent files, select only the commentary columns
df2 = pd.read_csv(file2)[ ['First Half Commentary', 'Second Half Commentary'] ]
df3 = pd.read_csv(file3)[ ['First Half Commentary', 'Second Half Commentary'] ]
df4 = pd.read_csv(file4)[ ['First Half Commentary', 'Second Half Commentary'] ]
print(df1.columns)
# 3. Combine side by side
combined = pd.concat([df1, df2, df3, df4], axis=1)
print("Final shape :", combined.shape)  # Should be (4180, 16)
display(combined.head())
print("After dropping 'Match URL' and 'Commentary':")
print("Final shape :", combined.shape)  # Should be (4180, 14)
display(combined.head())
#4. drop the Match URL column
combined.drop(columns=['Match URL'], inplace=True)
#4. Rename the commentary columns to indicate language
combined.columns = ['Season', 'Home Team', 'Away Team', 'Match Date Time','1H_Home_Shots_on_target', '1H_Away_Shots_on_target',
    '1H_Home_Shots_off_target', '1H_Away_Shots_off_target',
    '1H_Home_Blocked_Shots', '1H_Away_Blocked_Shots', '1H_Home_Total_shots',
    '1H_Away_Total_shots', '1H_Home_Ball_Possession',
    '1H_Away_Ball_Possession', '1H_Home_Fouls', '1H_Away_Fouls',
    '1H_Home_Yellow_Cards', '1H_Away_Yellow_Cards', '1H_Home_Offsides',
    '1H_Away_Offsides', '1H_Home_Corner_Kicks', '1H_Away_Corner_Kicks',
    '1H_Home_Throwins', '1H_Away_Throwins', '1H_Home_Free_Kicks',
    '1H_Away_Free_Kicks', '1H_Home_Goalkeeper_Saves',
    '1H_Away_Goalkeeper_Saves', '2H_Home_Shots_on_target',
    '2H_Away_Shots_on_target', '2H_Home_Shots_off_target',
    '2H_Away_Shots_off_target', '2H_Home_Blocked_Shots',
    '2H_Away_Blocked_Shots', '2H_Home_Total_shots', '2H_Away_Total_shots',
    '2H_Home_Ball_Possession', '2H_Away_Ball_Possession', '2H_Home_Fouls',
    '2H_Away_Fouls', '2H_Home_Yellow_Cards', '2H_Away_Yellow_Cards',
    '2H_Home_Offsides', '2H_Away_Offsides', '2H_Home_Corner_Kicks',
    '2H_Away_Corner_Kicks', '2H_Home_Throwins', '2H_Away_Throwins',
    '2H_Home_Free_Kicks', '2H_Away_Free_Kicks', '2H_Home_Goalkeeper_Saves',
    '2H_Away_Goalkeeper_Saves', 'FT_Home_Shots_on_target',
    'FT_Away_Shots_on_target', 'FT_Home_Shots_off_target',
    'FT_Away_Shots_off_target', 'FT_Home_Blocked_Shots',
    'FT_Away_Blocked_Shots', 'FT_Home_Total_shots', 'FT_Away_Total_shots',
    'FT_Home_Ball_Possession', 'FT_Away_Ball_Possession', 'FT_Home_Fouls',
    'FT_Away_Fouls', 'FT_Home_Yellow_Cards', 'FT_Away_Yellow_Cards',
    'FT_Home_Offsides', 'FT_Away_Offsides', 'FT_Home_Corner_Kicks',
    'FT_Away_Corner_Kicks', 'FT_Home_Throwins', 'FT_Away_Throwins',
    'FT_Home_Free_Kicks', 'FT_Away_Free_Kicks', 'FT_Home_Goalkeeper_Saves',
    'FT_Away_Goalkeeper_Saves',
    'First Half Commentary EN', 'Second Half Commentary EN',
    'First Half Commentary ES', 'Second Half Commentary ES',
    'First Half Commentary FR', 'Second Half Commentary FR',
    'First Half Commentary DE', 'Second Half Commentary DE'
]
# 5. Save if you want
output_file = 'flashscore_premier_league_combined_stats_commentary_all_languages.csv'
combined.to_csv(output_file, index=False)

print(f"Composite CSV successfully saved to {output_file}")


Index(['Season', 'Match URL', 'Home Team', 'Away Team', 'Match Date Time',
       '1H_Home_Shots_on_target', '1H_Away_Shots_on_target',
       '1H_Home_Shots_off_target', '1H_Away_Shots_off_target',
       '1H_Home_Blocked_Shots', '1H_Away_Blocked_Shots', '1H_Home_Total_shots',
       '1H_Away_Total_shots', '1H_Home_Ball_Possession',
       '1H_Away_Ball_Possession', '1H_Home_Fouls', '1H_Away_Fouls',
       '1H_Home_Yellow_Cards', '1H_Away_Yellow_Cards', '1H_Home_Offsides',
       '1H_Away_Offsides', '1H_Home_Corner_Kicks', '1H_Away_Corner_Kicks',
       '1H_Home_Throwins', '1H_Away_Throwins', '1H_Home_Free_Kicks',
       '1H_Away_Free_Kicks', '1H_Home_Goalkeeper_Saves',
       '1H_Away_Goalkeeper_Saves', '2H_Home_Shots_on_target',
       '2H_Away_Shots_on_target', '2H_Home_Shots_off_target',
       '2H_Away_Shots_off_target', '2H_Home_Blocked_Shots',
       '2H_Away_Blocked_Shots', '2H_Home_Total_shots', '2H_Away_Total_shots',
       '2H_Home_Ball_Possession', '2H_Away_Ball_Possession

,Season,Match URL,Home Team,Away Team,Match Date Time,1H_Home_Shots_on_target,1H_Away_Shots_on_target,1H_Home_Shots_off_target,1H_Away_Shots_off_target,1H_Home_Blocked_Shots,...,FT_Home_Goalkeeper_Saves,FT_Away_Goalkeeper_Saves,First Half Commentary,Second Half Commentary,First Half Commentary,Second Half Commentary,First Half Commentary,Second Half Commentary,First Half Commentary,Second Half Commentary
0,2013-2014,https://www.flashscore.com/match/football/nch9...,Cardiff,Chelsea,11.05. 07:00,2.0,2.0,2.0,6.0,2.0,...,5.0,3.0,45+3'\nWe have seen an attractive offensive ga...,"90+4'\nAccording to the statistics, the match ...",45+3'\nEstamos viendo un partido con un juego ...,"90+4'\nComo reflejan las estadísticas, el part...","45+3'\nJoli match offensif actuellement, et le...","90+4'\nSelon les statistiques, le match était ...",45+2'\nSo sollte Fußball immer sein. Voller Ac...,90+4'\nEs war ein sehr sehenswertes Spiel. Gro...
1,2013-2014,https://www.flashscore.com/match/football/tnbW...,Fulham,Crystal Palace,11.05. 07:00,2.0,4.0,4.0,2.0,2.0,...,4.0,3.0,45+2'\nThe game produced by the players until ...,"90+5'\nWe have seen a great game today, let's ...",45+2'\nEl juego no ha sido demasiado atractivo...,"90+5'\nGran partido el de hoy, ojalá que tambi...",45+2'\nLe jeu produit par les deux équipes n'é...,90+5'\nNous avons vu un beau match aujourd'hui...,45+3'\nWir sehen ein interessantes offensiv ge...,"90+4'\nWenn man auf die Statistiken schaut, wa..."
2,2013-2014,https://www.flashscore.com/match/football/fk2z...,Hull,Everton,11.05. 07:00,1.0,2.0,3.0,4.0,1.0,...,2.0,3.0,45+2'\nWe have been witnesses to an ordinary g...,90+3'\nThe performance from both sides could b...,45+2'\nEl encuentro está siendo flojo y carent...,90+3'\nPartido poco brillante. Sólo ha habido ...,NaN,NaN,45+2'\nDas Match war bisher nicht eins der att...,90+5'\nWir haben heute ein tolles Spiel gesehe...
3,2013-2014,https://www.flashscore.com/match/football/bNSG...,Liverpool,Newcastle,11.05. 07:00,2.0,2.0,1.0,3.0,2.0,...,1.0,3.0,45+3'\nIt wasn't the most inspiring 45 minutes...,"90+5'\nWe saw a few chances and nice plays, bu...",45+3'\nLos primeros 45 minutos fueron poco atr...,90+5'\nHemos visto algunas ocasiones y buenas ...,45+3'\nCe seront les 45 minutes que vous verre...,90+5'\nNous avons vu peu d'occasions et de mou...,45+3'\nDas waren jetzt nicht gerade die besten...,90+5'\nWir haben einige Chancen und nette Szen...
4,2013-2014,https://www.flashscore.com/match/football/0WEr...,Manchester City,West Ham,11.05. 07:00,3.0,0.0,7.0,1.0,4.0,...,0.0,5.0,45+2'\nNot the best game in the world so far b...,90+4'\nAll in all that was an entertaining 90 ...,45+2'\nNo hemos visto un gran partido hasta ah...,90+4'\n90 minutos de fútbol entretenidos con u...,"45+2'\nPas le meilleur match du monde, mais su...",90+4'\nAu final c'était 90 minutes de football...,45+3'\nDas waren jetzt nicht gerade die besten...,90+4'\nEs gab nicht besonders viele aufregende...


After dropping 'Match URL' and 'Commentary':
Final shape : (4180, 85)


,Season,Match URL,Home Team,Away Team,Match Date Time,1H_Home_Shots_on_target,1H_Away_Shots_on_target,1H_Home_Shots_off_target,1H_Away_Shots_off_target,1H_Home_Blocked_Shots,...,FT_Home_Goalkeeper_Saves,FT_Away_Goalkeeper_Saves,First Half Commentary,Second Half Commentary,First Half Commentary,Second Half Commentary,First Half Commentary,Second Half Commentary,First Half Commentary,Second Half Commentary
0,2013-2014,https://www.flashscore.com/match/football/nch9...,Cardiff,Chelsea,11.05. 07:00,2.0,2.0,2.0,6.0,2.0,...,5.0,3.0,45+3'\nWe have seen an attractive offensive ga...,"90+4'\nAccording to the statistics, the match ...",45+3'\nEstamos viendo un partido con un juego ...,"90+4'\nComo reflejan las estadísticas, el part...","45+3'\nJoli match offensif actuellement, et le...","90+4'\nSelon les statistiques, le match était ...",45+2'\nSo sollte Fußball immer sein. Voller Ac...,90+4'\nEs war ein sehr sehenswertes Spiel. Gro...
1,2013-2014,https://www.flashscore.com/match/football/tnbW...,Fulham,Crystal Palace,11.05. 07:00,2.0,4.0,4.0,2.0,2.0,...,4.0,3.0,45+2'\nThe game produced by the players until ...,"90+5'\nWe have seen a great game today, let's ...",45+2'\nEl juego no ha sido demasiado atractivo...,"90+5'\nGran partido el de hoy, ojalá que tambi...",45+2'\nLe jeu produit par les deux équipes n'é...,90+5'\nNous avons vu un beau match aujourd'hui...,45+3'\nWir sehen ein interessantes offensiv ge...,"90+4'\nWenn man auf die Statistiken schaut, wa..."
2,2013-2014,https://www.flashscore.com/match/football/fk2z...,Hull,Everton,11.05. 07:00,1.0,2.0,3.0,4.0,1.0,...,2.0,3.0,45+2'\nWe have been witnesses to an ordinary g...,90+3'\nThe performance from both sides could b...,45+2'\nEl encuentro está siendo flojo y carent...,90+3'\nPartido poco brillante. Sólo ha habido ...,NaN,NaN,45+2'\nDas Match war bisher nicht eins der att...,90+5'\nWir haben heute ein tolles Spiel gesehe...
3,2013-2014,https://www.flashscore.com/match/football/bNSG...,Liverpool,Newcastle,11.05. 07:00,2.0,2.0,1.0,3.0,2.0,...,1.0,3.0,45+3'\nIt wasn't the most inspiring 45 minutes...,"90+5'\nWe saw a few chances and nice plays, bu...",45+3'\nLos primeros 45 minutos fueron poco atr...,90+5'\nHemos visto algunas ocasiones y buenas ...,45+3'\nCe seront les 45 minutes que vous verre...,90+5'\nNous avons vu peu d'occasions et de mou...,45+3'\nDas waren jetzt nicht gerade die besten...,90+5'\nWir haben einige Chancen und nette Szen...
4,2013-2014,https://www.flashscore.com/match/football/0WEr...,Manchester City,West Ham,11.05. 07:00,3.0,0.0,7.0,1.0,4.0,...,0.0,5.0,45+2'\nNot the best game in the world so far b...,90+4'\nAll in all that was an entertaining 90 ...,45+2'\nNo hemos visto un gran partido hasta ah...,90+4'\n90 minutos de fútbol entretenidos con u...,"45+2'\nPas le meilleur match du monde, mais su...",90+4'\nAu final c'était 90 minutes de football...,45+3'\nDas waren jetzt nicht gerade die besten...,90+4'\nEs gab nicht besonders viele aufregende...


Composite CSV successfully saved to flashscore_premier_league_combined_stats_commentary_all_languages.csv
